# Corrected Hikaru baseline: 3,130 updates
Use all Hikaru training positions with a frozen Maia-3 5M backbone. Keep the existing chronological splits.
Training runs for exactly 3,130 successful optimizer updates, validating every 250 and at the end.
Select a GPU runtime. For a private repository, upload its ZIP in the setup cell.


In [ ]:
from pathlib import Path
import io, os, subprocess, sys, zipfile
from google.colab import files

workspace = Path('/content/chess-personas-work')
workspace.mkdir(exist_ok=True)
zip_path = Path('/content/Maia-Chess-starter.zip')
uploaded = {zip_path.name: zip_path.read_bytes()} if zip_path.exists() else files.upload()
archives = [name for name in uploaded if name.endswith('.zip')]
assert len(archives) == 1, 'Upload exactly one repository ZIP'
with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
    for member in archive.infolist():
        target = (workspace / member.filename).resolve()
        if not target.is_relative_to(workspace.resolve()):
            raise ValueError('Unsafe archive path')
    archive.extractall(workspace)
projects = [p.parent for p in workspace.rglob('pyproject.toml')
            if (p.parent / 'requirements-maia.txt').exists()]
assert len(projects) == 1, 'Expected one Chess Personas repository'
os.chdir(projects[0])
print('Repository:', Path.cwd())

def run(*args):
    command = [sys.executable, '-u', *map(str, args)]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

run('-m', 'pip', 'install', '-e', '.[train,dev]')
run('-m', 'pip', 'install', '-r', 'requirements-maia.txt')

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — switch to GPU for training')
run('-m', 'pytest', '-q')

## Download and inspect data
Public archive requests are sequential and cached. Increase the month count only
after the first run works. A full-history request is `--max-months 0`.
If preparing a second time, choose a new output directory to preserve old splits.

In [ ]:
import json
if not Path('data/processed/train.jsonl').exists():
    run('-m', 'chess_personas.cli', 'download', 'hikaru', 'fabianocaruana', '--max-months', '3')
    run('-m', 'chess_personas.cli', 'prepare', 'data/raw/*/*.json', '--output', 'data/processed')
print(Path('data/processed/audit.json').read_text())


## Run the corrected baseline
Preserve previous models by using a new output directory. Keep batch size 32,
learning rate 0.0003, seed 42 and the original model architecture. Validation
and a fixed training subset are scored with identical metrics. Timing baselines
are fit only on training data. The held-out test set is not used here.


In [ ]:
assert torch.cuda.is_available(), 'Select a GPU runtime before training'
RUN_DIR = Path('runs/hikaru-corrected-3130')
assert not RUN_DIR.exists(), 'Choose a new RUN_DIR to preserve previous work'
run('-m', 'chess_personas.cli', 'train', '--player', 'hikaru', '--output', RUN_DIR,
    '--max-per-player', '0', '--max-updates', '3130', '--eval-every', '250',
    '--batch-size', '32', '--learning-rate', '0.0003', '--seed', '42')


In [ ]:
summary = json.loads((RUN_DIR / 'run-summary.json').read_text())
assert summary['completed_updates'] == 3130
assert summary['unique_positions_seen'] == summary['training_positions']
print(json.dumps(summary, indent=2))
history = json.loads((RUN_DIR / 'metrics.json').read_text())
best = min(history, key=lambda item: item['selection_score'])
print('Selected validation metrics:', json.dumps(best['validation']['overall'], indent=2))


## Export checkpoints, metrics, baseline and prediction records


In [ ]:
import shutil
archive_path = shutil.make_archive('/content/hikaru-corrected-3130', 'zip', RUN_DIR)
files.download(archive_path)
